# 05 — Guardrail Metrics and Non-Inferiority Testing
**Prerequisites:** `04_primary_secondary_metrics.ipynb` (metrics taxonomy);
statistics_course/06_hypothesis_testing.ipynb (one-sided tests).
**Connects to:** `03_power_analysis.ipynb` (sizing for a fixed margin); `01_ab_testing_fundamentals.ipynb`.

## Narrative thread
```
Guardrail categories -> setting thresholds -> non-inferiority framing (one-sided tests, margins)
   -> worked case-study example
```

## Guardrail categories

Guardrail metrics answer "did we break something we're not even testing for," and are checked on
*every* experiment regardless of its hypothesis:

| Category | Examples | Failure mode it catches |
|---|---|---|
| **Latency / performance** | p50/p95/p99 page load time, time-to-interactive, API latency | A feature slows the product enough to hurt engagement even if it "wins" on the primary metric |
| **Trust / quality** | crash rate, error rate, complaint/unsubscribe rate, support-ticket rate | The feature technically improves the OEC while degrading the experience in a way users notice |
| **Organizational** | revenue at the company level, legal/compliance flags, brand metrics | Local win, global loss — a feature that helps one team's metric while damaging a company-wide constraint |

## Non-inferiority framing

Guardrails are *not* tested with a standard two-sided test — a two-sided test asks "is there any
difference," but for a guardrail the business question is asymmetric: **"is treatment not
meaningfully worse than control?"** This is a **non-inferiority test**, borrowed from clinical
trials methodology, with a pre-specified non-inferiority margin $\Delta > 0$:

$$H_0: \mu_{treatment} - \mu_{control} \le -\Delta \qquad \text{vs.} \qquad H_1: \mu_{treatment} - \mu_{control} > -\Delta$$

We reject $H_0$ (declare non-inferiority) when the **one-sided** $100(1-\alpha)\%$ confidence
bound for $\hat\tau = \bar Y_{treatment}-\bar Y_{control}$ lies entirely above $-\Delta$:

$$\hat{\tau} - z_{1-\alpha}\, SE(\hat\tau) > -\Delta$$

Choosing $\Delta$ is a business decision, not a statistical one — e.g. "we tolerate up to a 1%
relative regression in p95 latency" — and should be set *before* looking at the data.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Worked non-inferiority example: does a new checkout flow regress p95 latency? ──
np.random.seed(11)
n_per_arm = 20_000

# latency in ms, log-normal (typical for web latency metrics)
control_latency = np.random.lognormal(mean=np.log(400), sigma=0.35, size=n_per_arm)
# treatment is *slightly* slower (small true regression) but we want to confirm it's within tolerance
treatment_latency = np.random.lognormal(mean=np.log(408), sigma=0.35, size=n_per_arm)

mean_c, mean_t = control_latency.mean(), treatment_latency.mean()
se_c, se_t = control_latency.std(ddof=1) / np.sqrt(n_per_arm), treatment_latency.std(ddof=1) / np.sqrt(n_per_arm)
diff = mean_t - mean_c
se_diff = np.sqrt(se_c**2 + se_t**2)

alpha = 0.05
z_one_sided = stats.norm.ppf(1 - alpha)
lower_bound = diff - z_one_sided * se_diff   # one-sided lower confidence bound on the (possibly negative) effect

margin = 0.03 * mean_c   # tolerate up to a 3% relative regression in mean latency
print(f"Control mean latency: {mean_c:.1f} ms, Treatment mean latency: {mean_t:.1f} ms")
print(f"Observed difference (treatment - control): {diff:+.2f} ms  ({diff/mean_c:+.2%} relative)")
print(f"Non-inferiority margin (Delta): {margin:.2f} ms  (3% of control mean)")
print(f"One-sided 95% lower confidence bound on the difference: {lower_bound:+.2f} ms")

if lower_bound > -margin:
    print("=> Non-inferiority CONFIRMED: even the worst-case bound is within the tolerated margin.")
else:
    print("=> Non-inferiority NOT confirmed: latency regression could plausibly exceed the tolerated margin.")

Control mean latency: 426.3 ms, Treatment mean latency: 435.8 ms
Observed difference (treatment - control): +9.52 ms  (+2.23% relative)
Non-inferiority margin (Delta): 12.79 ms  (3% of control mean)
One-sided 95% lower confidence bound on the difference: +6.96 ms
=> Non-inferiority CONFIRMED: even the worst-case bound is within the tolerated margin.


## Setting thresholds in practice

- **Latency guardrails**: margins are often set from historical "known-safe" regressions (e.g.
  the largest regression ever shipped without a measurable drop in engagement), not an arbitrary
  round number.
- **Trust/quality guardrails**: often use a *strict* two-sided or even one-directional "must not
  increase at all" rule for severe outcomes (crash rate, legal complaints) rather than a tolerance
  margin — the margin for these is effectively zero.
- **Org guardrails**: usually monitored in aggregate across *all* running experiments (not just
  one), since the marginal effect of a single experiment on total revenue is tiny but the sum
  across dozens of concurrent experiments is not — this is closer to portfolio-level monitoring
  than a single-experiment test.

## Key takeaways

| Concept | Statement |
|---|---|
| Guardrail categories | Latency/performance, trust/quality, organizational |
| Non-inferiority test | One-sided test against a pre-specified margin $\Delta$, not a two-sided test against zero |
| Threshold setting | Business decision, ideally anchored in historical safe/unsafe regressions, set before the experiment |

## References

- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 6-7 (guardrail metrics), Ch. 17 (institutional memory / trust).
- Fabijan, A., Dmitriev, P., Olsson, H. H., & Bosch, J. (2017). The Benefits of Controlled Experimentation at Scale. *EUROMICRO SEAA*.
- Non-inferiority testing methodology follows standard clinical-trials practice (one-sided confidence bound vs. pre-specified margin); see also statistics_course/06_hypothesis_testing.ipynb for one-sided test mechanics.
